In [ ]:
# This notebook is generic and driven by a configuration file.
# The name of the configuration to use is passed in via a widget.

# 1. Create a widget to accept the configuration name.
dbutils.widgets.text("config_name", "", "Name of the config file in /config (e.g., ewm_ddl_ext)")

# 2. Retrieve the value from the widget.
config_name = dbutils.widgets.get("config_name")

if not config_name:
    raise ValueError("Widget 'config_name' must not be empty. Please provide a config file name.")

print(f"✅ Using configuration for: '{config_name}'")

In [ ]:
# This cell reads the shared configuration file and makes it available to the rest of the notebook.
import yaml

# --- CONFIGURATION ---
# The path is dynamically constructed based on the widget value.
config_path = f"../config/{config_name}.yml"

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Config file not found at path: {config_path}. Make sure the widget name matches the file name.")

# Extract specific sections for clarity
job_a_config = config['job_a']
storage_config = config['storage'] # New storage config
shared_config = config

print("✅ Configuration loaded successfully.")

In [ ]:
# ===================================================================
# Full Environment Cleanup Script (External Table Aware)
# ===================================================================

# --- Configuration (from config file) ---
target_table_name = shared_config['job_a_validation']['target_table_name_full']
control_table_schema = shared_config['control_table_schema']
base_raw_path = storage_config['base_raw_path']
base_checkpoint_path = storage_config['base_checkpoint_path']

# --- Dynamically determine paths ---
_, _, target_tbl_simple = target_table_name.rpartition('.')
external_table_path = f"{base_raw_path.rstrip('/')}/{target_tbl_simple}"
checkpoint_path = f"{base_checkpoint_path.rstrip('/')}/{target_tbl_simple}_stream"

# --- Control and Log Table Definitions ---
job_a_log_table = f"{control_table_schema}.job_a_execution_log"
stream_log_table = f"{control_table_schema}.stream_job_execution_log"
microbatch_log_table = f"{control_table_schema}.microbatch_execution_log"
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"

# --- Execution ---
print("Starting full environment cleanup...")

# 1. Drop the metastore table definition
print(f"Dropping table definition: {target_table_name}...")
spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")

# 2. **NEW**: Remove the physical data for the external table from cloud storage
print(f"Removing external table data at: {external_table_path}...")
try:
    dbutils.fs.rm(external_table_path, recurse=True)
    print(f"  - Successfully removed external data folder.")
except Exception as e:
    if "java.io.FileNotFoundException" in str(e) or "No such file or directory" in str(e):
        print("  - External data folder did not exist. Nothing to remove.")
    else:
        print(f"  - Warning: Failed to remove external data folder. It may need manual cleanup. Error: {e}")

# 3. Remove the streaming checkpoint directory
print(f"Removing checkpoint directory: {checkpoint_path}...")
try:
    dbutils.fs.rm(checkpoint_path, recurse=True)
    print("  - Checkpoint directory removed.")
except Exception as e:
    if "java.io.FileNotFoundException" in str(e) or "No such file or directory" in str(e):
        print("  - Checkpoint directory did not exist. Nothing to remove.")
    else:
        print(f"  - Warning: Failed to remove checkpoint directory. Error: {e}")

# 4. Drop all control and log tables (backward compatible)
print(f"Dropping control and log tables in schema: {control_table_schema}...")
spark.sql(f"DROP TABLE IF EXISTS {job_a_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {stream_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {microbatch_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {schema_transition_table}")
spark.sql(f"DROP TABLE IF EXISTS {schema_store_columns_table}")

# 5. Drop the control schema itself
print(f"Dropping control schema: {control_table_schema}...")
spark.sql(f"DROP SCHEMA IF EXISTS {control_table_schema} CASCADE")

print("\n✅ Cleanup Complete. The environment is now clean.")

In [ ]:
# ===================================================================================
# JOB A: UNIFIED SCHEMA ACTIVATOR & BACKFILL ORCHESTRATOR (External Table Aware)
# ===================================================================================
import requests
import json
import hashlib
import uuid
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from datetime import datetime

# ===================================================================================
# --- CONFIGURATION (from config file) ---
# ===================================================================================
ddl_url = job_a_config["ddl_url"]
source_data_path = job_a_config["source_data_path"]
unique_id_column = job_a_config["unique_id_column"]
triggering_event = job_a_config["triggering_event"]
control_table_schema = shared_config["control_table_schema"]
base_raw_path = storage_config['base_raw_path'] # NEW
# ===================================================================================

### Section 1: Initialization & Control Plane Bootstrapping ###
run_id = str(uuid.uuid4())
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
job_a_execution_log_table = f"{control_table_schema}.job_a_execution_log"
stream_log_table = f"{control_table_schema}.stream_job_execution_log"
microbatch_log_table = f"{control_table_schema}.microbatch_execution_log"

print("✅ Job A Configuration Initialized")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {control_table_schema}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {job_a_execution_log_table} (run_id STRING, run_timestamp TIMESTAMP, status STRING, ddl_location STRING, triggering_event STRING, old_schema_hash STRING, new_schema_hash STRING, message STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema_transition_table} (target_table_name STRING, schema_hash STRING, ddl_location STRING, event_type STRING, event_ts TIMESTAMP, backfill_status STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema_store_columns_table} (target_table_name STRING, contract_hash STRING, name_in_ddl STRING, physical_column_name STRING, type STRING, source_kind STRING, source_column STRING, json_key STRING, ddl_location STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {stream_log_table} (run_id STRING, r_src_id STRING, etl_job STRING, table_name STRING, start_tmstmp TIMESTAMP, end_tmstmp TIMESTAMP, load_status_cd STRING, message STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {microbatch_log_table} (microbatch_id STRING, run_id STRING, r_src_id STRING, etl_job STRING, table_name STRING, spark_batch_id LONG, start_tmstmp TIMESTAMP, end_tmstmp TIMESTAMP, row_count LONG, load_status_cd STRING, message STRING) USING DELTA")
print("   - All control tables created or verified.")

### Section 2: Core Utility Functions ###
def fetch_ddl_from_url(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def calculate_canonical_hash(ddl_json: dict) -> str:
    hash_inputs = sorted([f"{c['name']};{c['type']};{c['source_mapping']}" for c in ddl_json.get("columns", [])])
    return hashlib.md5("||".join(hash_inputs).encode("utf-8")).hexdigest()

def get_latest_active_hash(table_name: str, target_table_filter: str) -> str:
    if not spark.catalog.tableExists(table_name): return ""
    query = f"SELECT schema_hash FROM {table_name} WHERE target_table_name = '{target_table_filter}' AND event_type = 'ACTIVE' LIMIT 1"
    result = spark.sql(query).collect()
    return result[0]["schema_hash"] if result else ""

def sync_delta_schema(target_table: str, processed_ddl_df: F.DataFrame):
    all_physical_columns = processed_ddl_df.collect()
    try:
        spark.table(target_table).schema
    except AnalysisException as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
            # **NEW LOGIC FOR EXTERNAL TABLE**
            print(f"   - Table not found. Creating new EXTERNAL table '{target_table}' from DDL...")
            simple_table_name = target_table.split('.')[-1]
            table_location = f"{base_raw_path.rstrip('/')}/{simple_table_name}"
            print(f"   - Physical data will be stored at: {table_location}")

            columns_sql = [f"`{col['physical_column_name']}` {col['type'].upper()}" for col in all_physical_columns]
            if "_rescued_data" not in [c['physical_column_name'].lower() for c in all_physical_columns]:
                 columns_sql.append("`_rescued_data` STRING")
            
            create_sql = f"""CREATE TABLE {target_table} ({', '.join(columns_sql)}) \n                           USING DELTA \n                           LOCATION '{table_location}'"""
            spark.sql(create_sql)
            print("   - External table created successfully from DDL.")
            return
        else: raise e
    
    existing_columns_lower = {col.name.lower() for col in spark.table(target_table).schema.fields}
    new_columns_to_add = [col for col in all_physical_columns if col['physical_column_name'].lower() not in existing_columns_lower]
    if new_columns_to_add:
        add_cols_sql = [f"`{col['physical_column_name']}` {col['type'].upper()}" for col in new_columns_to_add]
        spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({', '.join(add_cols_sql)})")

def update_job_a_log(status: str, message: str, old_hash: str = "", new_hash: str = ""):
    clean_message = message.replace("'", "''")
    spark.sql(f"MERGE INTO {job_a_execution_log_table} t USING (SELECT '{run_id}' as run_id) s ON t.run_id = s.run_id WHEN MATCHED THEN UPDATE SET status = '{status}', message = '{clean_message}', old_schema_hash = '{old_hash}', new_schema_hash = '{new_hash}' WHEN NOT MATCHED THEN INSERT (run_id, run_timestamp, status, ddl_location, triggering_event, old_schema_hash, new_schema_hash, message) VALUES ('{run_id}', current_timestamp(), '{status}', '{ddl_url}', '{triggering_event}', '{old_hash}', '{new_hash}', '{clean_message}')")
    print(f"   - Log updated for run {run_id} with status: {status}")

### Section 3: Main Processing & Backfill Logic ###
try:
    update_job_a_log("RUNNING", "Job started.")
    ddl_data = fetch_ddl_from_url(ddl_url)
    full_target_table = ddl_data.get("table_name")
    if not full_target_table: raise ValueError(f"DDL from '{ddl_url}' must contain a 'table_name' property.")
    
    simple_table_name = full_target_table.split('.')[-1]
    
    new_hash = calculate_canonical_hash(ddl_data)
    
    columns_data = []
    for col in ddl_data.get("columns", []):
        if mapping := col.get("source_mapping"):
            parts = mapping.split('.', 2)
            source_kind, source_column, json_key = parts[0].upper(), None, None
            if source_kind == 'JSON':
                if len(parts) < 3: raise ValueError(f"Invalid JSON mapping: {mapping}")
                source_column, json_key = parts[1], parts[2]
            elif len(parts) > 1: source_column = parts[1]
            columns_data.append((col['name'], col['name'], col['type'], source_kind, source_column, json_key))
    
    processed_ddl_df = spark.createDataFrame(columns_data, "name_in_ddl STRING, physical_column_name STRING, type STRING, source_kind STRING, source_column STRING, json_key STRING")
    latest_known_hash = get_latest_active_hash(schema_transition_table, simple_table_name)

    if new_hash == latest_known_hash:
        update_job_a_log("SUCCESS", "No schema change detected.", old_hash=latest_known_hash, new_hash=new_hash)
        print("✅ No schema change detected. Job finished.")
    else:
        print(f"Schema change detected. New Hash: {new_hash}")
        old_schema_hash = latest_known_hash
        sync_delta_schema(full_target_table, processed_ddl_df)
        
        blueprint_df = processed_ddl_df.withColumn("target_table_name", F.lit(simple_table_name)).withColumn("contract_hash", F.lit(new_hash)).withColumn("ddl_location", F.lit(ddl_url))
        blueprint_df.write.format("delta").mode("overwrite").option("replaceWhere", f"target_table_name = '{simple_table_name}' AND contract_hash = '{new_hash}'").saveAsTable(schema_store_columns_table)

        if old_schema_hash:
            print("\n   - Beginning backfill process...")
            spark.sql(f"UPDATE {schema_transition_table} SET event_type = 'ARCHIVED' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{old_schema_hash}'")
            spark.sql(f"INSERT INTO {schema_transition_table} VALUES ('{simple_table_name}', '{new_hash}', '{ddl_url}', 'ACTIVE', current_timestamp(), 'BACKFILL_IN_PROGRESS')")

            schema_store_df = spark.table(schema_store_columns_table).filter(f"target_table_name = '{simple_table_name}'")
            new_schema_bp = schema_store_df.filter(f"contract_hash = '{new_hash}'")
            old_schema_bp = schema_store_df.filter(f"contract_hash = '{old_schema_hash}'")
            new_cols = new_schema_bp.join(old_schema_bp, on="physical_column_name", how="left_anti").filter(~F.col("source_kind").startswith("PIPELINE")).collect()
            
            if not new_cols:
                print("   - No new columns with source mappings found. Backfill not required.")
                spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
            else:
                rows_to_backfill_df = spark.table(full_target_table).filter(f"schema_hash_at_ingest = '{old_schema_hash}'")
                if rows_to_backfill_df.isEmpty():
                     print("   - No rows found with the old schema hash. Backfill not required.")
                     spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
                else:
                    print("   - Backfilling new columns for existing data...")
                    rows_to_backfill_df.select(unique_id_column).distinct().createOrReplaceTempView("keys_to_backfill")
                    source_df = spark.read.format("parquet").load(source_data_path)
                    backfill_source_df = source_df.join(spark.table("keys_to_backfill"), unique_id_column, "inner")
                    
                    processed_backfill_df = backfill_source_df
                    json_cols_to_process = [r for r in new_schema_bp.collect() if r['source_kind'] == 'JSON']
                    for r in json_cols_to_process:
                        processed_backfill_df = processed_backfill_df.withColumn(r['physical_column_name'], F.get_json_object(F.col(r['source_column']), f"$.{r['json_key']}"))
                    
                    final_cols_for_merge = [unique_id_column] + [c['physical_column_name'] for c in new_cols]
                    backfill_update_view = processed_backfill_df.select(final_cols_for_merge)
                    backfill_update_view.createOrReplaceTempView("backfill_source_view")
                    
                    update_clauses = [f"target.`{c['physical_column_name']}` = source.`{c['physical_column_name']}`" for c in new_cols]
                    update_clauses.append(f"target.schema_hash_at_ingest = '{new_hash}'")
                    
                    merge_sql = f"""MERGE INTO {full_target_table} AS target \n                                     USING backfill_source_view AS source \n                                     ON target.{unique_id_column} = source.{unique_id_column} \n                                     WHEN MATCHED AND target.schema_hash_at_ingest = '{old_schema_hash}' THEN UPDATE SET {', '.join(update_clauses)}"""
                    print("   - Executing backfill MERGE statement...")
                    spark.sql(merge_sql)
                    print("   - Backfill MERGE complete.")
                    spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
        else:
            print("   - No previous active schema found. This is the first activation.")
            spark.sql(f"INSERT INTO {schema_transition_table} VALUES ('{simple_table_name}', '{new_hash}', '{ddl_url}', 'ACTIVE', current_timestamp(), 'NOT_APPLICABLE')")
            
        update_job_a_log("SUCCESS", "New schema activated successfully.", old_hash=old_schema_hash, new_hash=new_hash)
        print("\n✅ New schema activated successfully.")
except Exception as e:
    import traceback
    error_message = f"FATAL ERROR in Job A run {run_id}: {traceback.format_exc()}"
    print(error_message)
    try:
        update_job_a_log("FAIL", error_message)
    except Exception as log_e:
        print(f"CRITICAL: Failed to write to execution log. Error: {log_e}")
    raise e